## ÉTAPE 5 — Comparer plusieurs régions
Objectif : production de maïs par région 1961-2007


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


## 1. CHARGER LES DONNÉES


In [ ]:
df = pd.read_csv("JEU_DE_DONNEE.csv")
print(f"Dataset chargé : {df.shape[0]:,} lignes\n")


## 2. CHOISIR LES RÉGIONS À COMPARER


In [ ]:
# Voici toutes les régions disponibles dans le dataset :
# "Africa +", "Americas +", "Asia +", "Europe +",
# "Northern America +", "South America +",
# "Eastern Asia +", "Southern Asia +", "Oceania +"...
# On choisit les 5 grandes régions continentales

regions = [
    "Northern America +",
    "Asia +",
    "Europe +",
    "South America +",
    "Africa +"
]


## 3. FILTRER LES DONNÉES


In [ ]:
df_maize = df[
    (df["category"] == "maize") &
    (df["element"] == "Production Quantity") &
    (df["country_or_area"].isin(regions))   # isin() filtre sur une liste de valeurs
].dropna(subset=["value"]).copy()

# Convertir en millions de tonnes
df_maize["valeur_millions"] = df_maize["value"] / 1_000_000

print(f"Lignes après filtrage : {df_maize.shape[0]}")
print(f"Régions trouvées : {df_maize['country_or_area'].unique()}\n")


## 4. PIVOT TABLE — Restructurer pour le graphique


In [ ]:
# pivot_table() transforme le tableau :
# - chaque année devient une ligne
# - chaque région devient une colonne
# C'est la structure idéale pour tracer plusieurs courbes

pivot = df_maize.pivot_table(
    index="year",           # les lignes = années
    columns="country_or_area",  # les colonnes = régions
    values="valeur_millions",   # les valeurs = production
    aggfunc="sum"           # si doublons : additionner
)

# Renommer les colonnes pour enlever le " +"
pivot.columns = [col.replace(" +", "") for col in pivot.columns]

print("--- Aperçu du tableau pivot ---")
print(pivot.head(5).to_string())
print()


## 5. GRAPHIQUE — COURBES PAR RÉGION


In [ ]:
couleurs = {
    "Northern America" : "#2563EB",  # bleu
    "Asia"             : "#DC2626",  # rouge
    "Europe"           : "#16A34A",  # vert
    "South America"    : "#D97706",  # orange
    "Africa"           : "#7C3AED",  # violet
}

fig, ax = plt.subplots(figsize=(14, 7))

for region in pivot.columns:
    ax.plot(
        pivot.index,
        pivot[region],
        label=region,
        linewidth=2.5,
        marker="o",
        markersize=3.5,
        color=couleurs.get(region, "gray")
    )

# Mise en forme
ax.set_title(
    "Production de maïs par région (1961 – 2007)",
    fontsize=16, fontweight="bold", pad=15
)
ax.set_xlabel("Année", fontsize=12)
ax.set_ylabel("Production (millions de tonnes)", fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.0f} Mt"))
ax.set_xlim(1961, 2007)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(fontsize=11, loc="upper left")

plt.tight_layout()
plt.savefig("graphique_maize_regions.png", dpi=150, bbox_inches="tight")
print("Graphique 1 sauvegardé : 'graphique_maize_regions.png' ✓")
plt.show()


## 6. GRAPHIQUE 2 — BARRES EMPILÉES PAR DÉCENNIE


In [ ]:
# Regrouper par décennie pour simplifier la comparaison
df_maize["decennie"] = (df_maize["year"] // 10 * 10).astype(int)

pivot_dec = df_maize.pivot_table(
    index="decennie",
    columns="country_or_area",
    values="valeur_millions",
    aggfunc="sum"
)
pivot_dec.columns = [col.replace(" +", "") for col in pivot_dec.columns]

fig2, ax2 = plt.subplots(figsize=(10, 6))

pivot_dec.plot(
    kind="bar",
    stacked=True,               # barres empilées
    ax=ax2,
    color=list(couleurs.values()),
    width=0.6
)

ax2.set_title(
    "Production totale de maïs par décennie et région",
    fontsize=14, fontweight="bold", pad=12
)
ax2.set_xlabel("Décennie", fontsize=11)
ax2.set_ylabel("Production totale (millions de tonnes)", fontsize=11)
ax2.set_xticklabels(
    [f"{int(d)}s" for d in pivot_dec.index],
    rotation=0
)
ax2.legend(fontsize=10, loc="upper left")
ax2.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("graphique_maize_decennies.png", dpi=150, bbox_inches="tight")
print("Graphique 2 sauvegardé : 'graphique_maize_decennies.png' ✓")
plt.show()


## 7. TABLEAU RÉCAPITULATIF FINAL


In [ ]:
print("\n--- Production totale par région (1961-2007) ---")
total_par_region = (
    df_maize.groupby("country_or_area")["valeur_millions"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
total_par_region.columns = ["Région", "Production totale (Mt)"]
total_par_region["Région"] = total_par_region["Région"].str.replace(" +", "", regex=False)
total_par_region.index = total_par_region.index + 1
total_par_region.index.name = "Rang"
print(total_par_region.to_string())

# Croissance de la région numéro 1 (Amérique du Nord)
amer = pivot["Northern America"]
croissance = ((amer.iloc[-1] - amer.iloc[0]) / amer.iloc[0]) * 100
print(f"\nCroissance Northern America 1961→2007 : +{croissance:.0f}%")
